In [0]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, when


#  Chemin des fichiers CSV bruts
raw_path = "/Volumes/workspace/default/data/extracted_v/*.csv"
df_raw = spark.read.csv(raw_path, header=True, inferSchema=True)

print(f"schema du dataframe {df_raw.printSchema()}")
print(f"Nombre de lignes brutes : {df_raw.count()}")
print(f"Colonnes : {df_raw.columns}")



In [0]:
display(df_raw.limit(5))

In [0]:

# Calcul du nombre de nulls par colonne
null_counts = df_raw.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_raw.columns
])

# Transformer en format vertical (colonne / nombre_nulls)
null_counts_long = null_counts.selectExpr(
    "stack({0}, {1}) as (column_name, null_count)".format(
        len(df_raw.columns),
        ", ".join([f"'{c}', {c}" for c in df_raw.columns])
    )
)

# Garder seulement celles avec null_count > 0
null_columns_only = null_counts_long.filter(col("null_count") > 0)

null_columns_only.show(truncate=False)

In [0]:
# Doublons
df_raw.count() - df_raw.dropDuplicates().count()

In [0]:
# Valeurs négatives
df_raw.filter(df_raw["annual_avg_emplvl"] < 0).count()

###Nettoyage


In [0]:
from pyspark.sql import functions as F
# 1 Drop disclosure columns
to_drop = [
    c for c in [
        "disclosure_code",
        "lq_disclosure_code",
        "oty_disclosure_code"
    ] if c in df_raw.columns
]

df = df_raw.drop(*to_drop)

# 2 Filter agglvl_code = 73
if "agglvl_code" not in df.columns:
    raise ValueError("Column agglvl_code not found.")

df = df.filter(F.col("agglvl_code") == 73)

# 3 Keep own_code + Create sector_type
if "own_code" not in df.columns:
    raise ValueError("Column own_code not found.")

df = df.withColumn(
    "sector_type",
    F.when(F.col("own_code") == 1, "Private")
     .when(F.col("own_code").isin(2,3,5), "Public")
     .otherwise("Other")
)

# validation
print("Rows after filter:", df.count())
df.groupBy("sector_type").count().show()
df.groupBy("year").count().orderBy("year").show()

In [0]:

# Calcul du nombre de nulls par colonne
null_counts = df_raw.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_raw.columns
])

# Transformer en format vertical (colonne / nombre_nulls)
null_counts_long = null_counts.selectExpr(
    "stack({0}, {1}) as (column_name, null_count)".format(
        len(df_raw.columns),
        ", ".join([f"'{c}', {c}" for c in df_raw.columns])
    )
)

# Garder seulement celles avec null_count > 0
null_columns_only = null_counts_long.filter(col("null_count") > 0)

null_columns_only.show(truncate=False)

In [0]:
df_raw.count() - df_raw.dropDuplicates().count()